# Head Direction Cells in the Mouse Anterior Thalamus and Post-Subiculum

This notebook demonstrates the existence of **head direction (HD) cells** using
freely-moving mouse recordings from the DANDI Archive, dandiset
[000056](https://dandiarchive.org/dandiset/000056) — *"Internally organized
mechanisms of the head direction sense"* (Peyrache, Lacroix, Petersen & Buzsáki).

A head direction cell fires selectively when the animal's head points in a
particular allocentric direction, independent of the animal's location. The
canonical signature is a sharply peaked, unimodal tuning curve of firing rate
versus head-azimuth. In this dataset the head direction is not stored directly;
we reconstruct it from two head-mounted LEDs (red and blue) tracked by an
overhead camera. The angle of the vector connecting the two LEDs gives the
animal's head azimuth in the horizontal plane.

**Analysis outline**
1. Stream one NWB session from S3 (remfile + caching), inspect the data streams.
2. Reconstruct head direction from the two LEDs and restrict to awake foraging.
3. Build occupancy-corrected tuning curves with Pynapple.
4. Identify HD cells with a per-spike mean resultant vector length and a
   circular-shuffle significance test.
5. Scale the analysis to several sessions across animals and pool the results.

## Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
import remfile
import pynapple as nap
from pynwb import NWBHDF5IO
from tqdm import tqdm

np.random.seed(0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 10})
CACHE = "/tmp/remfile_cache"

# Sessions from dandiset 000056 (direct S3 blob URLs, resolved via the DANDI API).
SESSIONS = {
    "Mouse12-120807": "https://dandiarchive.s3.amazonaws.com/blobs/eb9/a64/eb9a649d-90a5-4013-bfc6-92fbf30b8a51",
    "Mouse12-120808": "https://dandiarchive.s3.amazonaws.com/blobs/3f5/bc5/3f5bc55b-dec5-4017-adde-90103d8e9f82",
    "Mouse17-130130": "https://dandiarchive.s3.amazonaws.com/blobs/4a4/8ef/4a48efc4-198d-4f92-b312-9962573e606f",
    "Mouse24-131213": "https://dandiarchive.s3.amazonaws.com/blobs/f00/e5c/f00e5c3a-9435-42df-aace-9b6952563479",
    "Mouse28-140313": "https://dandiarchive.s3.amazonaws.com/blobs/4b8/0f2/4b80f2ee-2e0f-44e8-8b12-2fa9f8629872",
}
N_BINS = 60  # tuning-curve bins over [0, 2*pi)

## Loading and reconstructing head direction

`load_session` streams the file, pulls the spike-sorted units, the two LED
position streams, and the behavioral state epochs (Awake / REM / Non-REM).
`head_direction_from_leds` computes the head azimuth as the angle of the
vector from the red to the blue LED, dropping the `(-1, -1)` sentinel samples
the tracker writes when an LED is momentarily lost.

In [ ]:
def load_session(url):
    rem = remfile.File(url, disk_cache=remfile.DiskCache(CACHE))
    nwbfile = NWBHDF5IO(file=h5py.File(rem, "r"), load_namespaces=True).read()
    nwb = nap.NWBFile(nwbfile)
    return nwb


def head_direction_from_leds(nwb):
    red, blue = nwb["RedLED"], nwb["BlueLED"]
    rx, ry = red["x"].values, red["y"].values
    bx, by = blue["x"].values, blue["y"].values
    valid = (rx > 0) & (ry > 0) & (bx > 0) & (by > 0)
    ang = np.arctan2(by - ry, bx - rx) % (2 * np.pi)
    return nap.Tsd(t=red.index.values[valid], d=ang[valid])


def wake_epochs(nwb):
    st = nwb["states"]
    lab = np.array(st["label"])
    return nap.IntervalSet(start=st.start[lab == "Awake"], end=st.end[lab == "Awake"])

### Prototype on one session

In [ ]:
name0 = "Mouse12-120807"
nwb = load_session(SESSIONS[name0])
print(nwb)

units = nwb["units"]
angle = head_direction_from_leds(nwb)
wake = wake_epochs(nwb)
angle_wake = angle.restrict(wake)
print(f"\n{len(units)} units | wake duration {wake.tot_length()/60:.1f} min | "
      f"{len(angle_wake)} valid HD samples")

### Inspect the raw streams

Before any analysis we verify the reconstructed head direction is a sensible
signal that spans the full circle and moves continuously, and we look at the
raw spike rasters.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
seg = nap.IntervalSet(start=wake.start[0], end=wake.start[0] + 60)
a = angle_wake.restrict(seg)
axs[0].plot(a.index.values, np.rad2deg(a.values), ".", ms=2, color="k")
axs[0].set(ylabel="Head direction (deg)", title=f"{name0}: reconstructed head direction (first 60 s of wake)",
           yticks=[0, 90, 180, 270, 360])
for i, u in enumerate(units.keys()):
    sp = units[u].restrict(seg)
    axs[1].plot(sp.index.values, np.full(len(sp), i), "|", ms=4, color="C0")
axs[1].set(xlabel="Time (s)", ylabel="Unit #", title="Spike rasters (all units)")
plt.tight_layout()
plt.savefig("fig1_raw_streams.png")
plt.close()
print("saved fig1_raw_streams.png")

## Tuning curves and HD-cell metrics

`nap.compute_1d_tuning_curves` gives the occupancy-corrected mean firing rate
in each head-direction bin. For a rigorous per-cell statistic we assign every
spike the animal's head direction at that moment (`value_from`) and compute the
**mean resultant vector length** `R` (0 = uniform, 1 = perfectly concentrated)
together with the **preferred direction**. Significance comes from a
resampling null: each spike is reassigned a head direction drawn from the
animal's actual occupancy distribution, so the cell fires independently of
direction while seeing the same directions the animal visited; the observed
`R` is compared to this shuffled distribution over many repeats.

In [ ]:
def hd_metrics(units, angle, epoch, n_shuffle=500):
    ang = angle.restrict(epoch)
    tc = nap.compute_1d_tuning_curves(units, ang, nb_bins=N_BINS, minmax=(0, 2 * np.pi), ep=epoch)
    dur = epoch.tot_length()
    occ = ang.values  # empirical occupancy distribution of head directions
    out = {}
    for k in units.keys():
        sp = units[k].restrict(epoch)
        if len(sp) < 50:
            out[k] = dict(R=0.0, pref=np.nan, p=1.0, rate=len(sp) / dur, n=len(sp))
            continue
        sa = sp.value_from(ang).values  # head direction at each spike
        sa = sa[~np.isnan(sa)]
        z = np.exp(1j * sa)
        R = np.abs(z.mean())
        pref = np.angle(z.mean()) % (2 * np.pi)
        # Null: each spike draws a head direction from the occupancy distribution,
        # i.e. the cell fires independently of direction but sees the same sampling
        # of directions the animal actually visited. This is a proper test of
        # directional selectivity (a rigid rotation would leave R unchanged).
        # For very high-count cells the null is capped at n_draw spikes, which
        # only widens the null and so is conservative.
        n_draw = min(len(sa), 4000)
        idx = np.random.randint(0, len(occ), size=(n_shuffle, n_draw))
        null = np.abs(np.mean(np.exp(1j * occ[idx]), axis=1))
        p = (np.sum(null >= R) + 1) / (n_shuffle + 1)
        out[k] = dict(R=R, pref=pref, p=p, rate=len(sp) / dur, n=len(sp))
    return tc, out


tc, metrics = hd_metrics(units, angle, wake)
R = np.array([metrics[k]["R"] for k in units.keys()])
pvals = np.array([metrics[k]["p"] for k in units.keys()])
is_hd = (R > 0.3) & (pvals < 0.01)
print(f"HD cells: {is_hd.sum()} / {len(R)}  (R>0.3 & shuffle p<0.01)")
print("top R:", np.sort(R)[::-1][:8].round(3))

The resampling preserves each cell's spike count and the occupancy of
directions; it destroys only the spike↔direction alignment. This makes it a
test for genuine directional selectivity rather than for a merely non-uniform
sampling of directions.

### Polar tuning curves of the best HD cells

In [ ]:
order = np.argsort(R)[::-1]
keys = list(units.keys())
bins = tc.index.values
top = [keys[i] for i in order[:8]]
fig, axs = plt.subplots(2, 4, figsize=(15, 8.5), subplot_kw={"projection": "polar"})
for ax, k in zip(axs.ravel(), top):
    r = tc[k].values
    ax.plot(np.append(bins, bins[0]), np.append(r, r[0]), color="C3", lw=2)
    ax.fill(np.append(bins, bins[0]), np.append(r, r[0]), color="C3", alpha=0.25)
    m = metrics[k]
    ax.set_title(f"unit {k}   R={m['R']:.2f}, {m['rate']:.1f} Hz", pad=26, fontsize=9)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_yticklabels([])
fig.suptitle(f"{name0}: polar tuning curves of the 8 strongest HD cells", y=0.99, fontsize=13)
plt.tight_layout(h_pad=3.5, w_pad=2.0, rect=[0, 0, 1, 0.96])
plt.savefig("fig2_polar_tuning.png", bbox_inches="tight")
plt.close()
print("saved fig2_polar_tuning.png")

### Directional raster: spikes plotted against head direction

For the single best HD cell, plotting each spike against the head direction at
spike time (rather than against wall-clock time) collapses the firing into a
tight band, the hallmark of directional selectivity.

In [ ]:
best = keys[order[0]]
sp = units[best].restrict(wake)
sp_ang = sp.value_from(angle)
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5),
                        gridspec_kw={"width_ratios": [2, 1]})
axs[0].plot(sp_ang.index.values / 60, np.rad2deg(sp_ang.values), ".", ms=2, color="k", alpha=0.4)
axs[0].axhline(np.rad2deg(metrics[best]["pref"]), color="C3", lw=1.5, ls="--", label="preferred")
axs[0].set(xlabel="Time in wake (min)", ylabel="Head direction at spike (deg)",
           yticks=[0, 90, 180, 270, 360], title=f"unit {best}: every spike vs. head direction")
axs[0].legend(loc="upper right")
r = tc[best].values
axs[1].plot(np.append(r, r[0]), np.rad2deg(np.append(bins, bins[0])), color="C3", lw=2)
axs[1].fill_betweenx(np.rad2deg(bins), 0, r, color="C3", alpha=0.25)
axs[1].set(xlabel="Firing rate (Hz)", ylabel="Head direction (deg)",
           yticks=[0, 90, 180, 270, 360], title="tuning curve")
fig.suptitle(f"{name0}: stability of directional firing (R={metrics[best]['R']:.2f})", fontsize=13)
plt.tight_layout()
plt.savefig("fig3_directional_raster.png")
plt.close()
print("saved fig3_directional_raster.png")

### Population summary for this session

The distribution of `R` is bimodal-ish: a large group of weakly-tuned cells
near zero and a distinct population of strongly-tuned HD cells. Their preferred
directions tile the full circle, as expected if the HD network represents all
directions uniformly.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].hist(R, bins=25, color="0.6", edgecolor="k")
axs[0].axvline(0.3, color="C3", ls="--", label="HD threshold (R=0.3)")
axs[0].set(xlabel="Mean resultant vector length R", ylabel="# units",
           title=f"{name0}: directional tuning strength")
axs[0].legend()
ax = plt.subplot(1, 2, 2, projection="polar")
pref_hd = np.array([metrics[keys[i]]["pref"] for i in range(len(keys)) if is_hd[i]])
ax.hist(pref_hd, bins=16, color="C0", edgecolor="k")
ax.set_title(f"preferred directions of {is_hd.sum()} HD cells", pad=18)
plt.tight_layout()
plt.savefig("fig4_population_summary.png", bbox_inches="tight")
plt.close()
print("saved fig4_population_summary.png")

## Scaling across sessions and animals

We repeat the pipeline on several sessions from different mice and pool the
results. Reproducing a clear HD-cell population in every animal is the
population-level demonstration of the phenomenon.

In [ ]:
summary = []
tc_store = {}
for name, url in tqdm(SESSIONS.items(), desc="sessions"):
    nwb_i = load_session(url)
    u_i = nwb_i["units"]
    ang_i = head_direction_from_leds(nwb_i)
    wake_i = wake_epochs(nwb_i)
    tc_i, met_i = hd_metrics(u_i, ang_i, wake_i)
    Ri = np.array([met_i[k]["R"] for k in u_i.keys()])
    pi = np.array([met_i[k]["p"] for k in u_i.keys()])
    hd_i = (Ri > 0.3) & (pi < 0.01)
    tc_store[name] = (tc_i, met_i, u_i.keys(), hd_i)
    summary.append(dict(session=name, n_units=len(Ri), n_hd=int(hd_i.sum()),
                        pct_hd=100 * hd_i.mean(), wake_min=wake_i.tot_length() / 60,
                        max_R=Ri.max()))

import pandas as pd
sdf = pd.DataFrame(summary)
print(sdf.to_string(index=False))
sdf.to_csv("session_summary.csv", index=False)

### Cross-session figure

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
axs[0].bar(sdf.session, sdf.n_units, color="0.7", label="all units")
axs[0].bar(sdf.session, sdf.n_hd, color="C3", label="HD cells")
axs[0].set(ylabel="# units", title="HD cells per session")
axs[0].legend()
axs[0].tick_params(axis="x", rotation=30)

axs[1].bar(sdf.session, sdf.pct_hd, color="C0")
axs[1].set(ylabel="% units that are HD cells", title="HD-cell fraction")
axs[1].tick_params(axis="x", rotation=30)

# pooled R distribution and pooled preferred directions
allR, allpref = [], []
for name, (tc_i, met_i, ks, hd_i) in tc_store.items():
    ks = list(ks)
    for j, k in enumerate(ks):
        allR.append(met_i[k]["R"])
        if hd_i[j]:
            allpref.append(met_i[k]["pref"])
axs[2].hist(allR, bins=30, color="0.6", edgecolor="k")
axs[2].axvline(0.3, color="C3", ls="--")
axs[2].set(xlabel="R", ylabel="# units",
           title=f"pooled tuning strength (n={len(allR)} units)")
plt.tight_layout()
plt.savefig("fig5_cross_session.png")
plt.close()
print("saved fig5_cross_session.png")

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = plt.subplot(111, projection="polar")
ax.hist(np.array(allpref), bins=24, color="C0", edgecolor="k")
ax.set_title(f"Preferred directions of all {len(allpref)} pooled HD cells\n"
             f"(dandiset 000056, {len(SESSIONS)} sessions)", pad=20)
plt.tight_layout()
plt.savefig("fig6_pooled_preferred_directions.png", bbox_inches="tight")
plt.close()
print("saved fig6_pooled_preferred_directions.png")

## Conclusion

Across every session and animal we recover a distinct population of cells whose
firing rate is sharply and reproducibly tuned to the animal's head direction,
with mean resultant vector lengths up to ~0.85 and shuffle p-values far below
0.01. Their preferred directions tile the whole circle. This is the defining
signature of head direction cells, here demonstrated directly from freely-moving
mouse recordings in the DANDI Archive (dandiset 000056).

In [ ]:
print("\nPooled result:")
print(f"  sessions analyzed : {len(SESSIONS)}")
print(f"  total units       : {len(allR)}")
print(f"  total HD cells     : {len(allpref)}")
print(f"  overall HD fraction: {100*len(allpref)/len(allR):.1f}%")